In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

In [2]:
import json
import numpy as np
from src.data_loader import load_raw_data
from src.preprocessing import clean_raw_data, chronological_split, fit_scaler, apply_scaler, inverse_transform_column
from src.feature_engineering import build_features
from src.sequence_builder import create_sequences_for_all_splits
from src.evaluation.metrics import evaluate_all
from src.config import MODEL_FEATURE_COLUMNS, TARGET_COL, DEFAULT_LOOKBACK, FORECAST_HORIZON

DEV_LOOKBACK = 72  # temporary, for fast iteration — NOT the final 168

df_raw = load_raw_data()
df_clean = clean_raw_data(df_raw)
df_features = build_features(df_clean)
splits = chronological_split(df_features)

scale_columns = MODEL_FEATURE_COLUMNS + [TARGET_COL]
scaler = fit_scaler(splits.train, scale_columns)

train_scaled = apply_scaler(splits.train, scaler, scale_columns)
val_scaled = apply_scaler(splits.val, scaler, scale_columns)
test_scaled = apply_scaler(splits.test, scaler, scale_columns)

sequences = create_sequences_for_all_splits(train_scaled, val_scaled, test_scaled, MODEL_FEATURE_COLUMNS)
X_train, y_train_scaled = sequences["train"]
X_val, y_val_scaled = sequences["val"]
X_test, y_test_scaled = sequences["test"]

print("Ready:", X_train.shape, X_val.shape, X_test.shape)

2026-09-13 21:38:24 | INFO     | src.data_loader | Loading raw dataset from C:\Users\Ram\OneDrive\Desktop\energy-demand-lstm\data\raw\continuous_dataset.csv
2026-09-13 21:38:26 | INFO     | src.data_loader | Loaded raw dataset: 48048 rows, 17 columns
2026-09-13 21:38:26 | INFO     | src.preprocessing | Cleaning complete: 48048 rows, 4 flagged as diagnostic extreme events (|z| > 4.0)
2026-09-13 21:38:26 | INFO     | src.feature_engineering | Added national weather averages: temp_avg, humidity_avg, precip_avg, wind_avg
2026-09-13 21:38:26 | INFO     | src.feature_engineering | Added calendar features and cyclical encodings
2026-09-13 21:38:27 | INFO     | src.feature_engineering | Added lag features: ['lag_1', 'lag_24', 'lag_48', 'lag_168']
2026-09-13 21:38:27 | INFO     | src.feature_engineering | Added rolling features: rolling_mean_24, rolling_std_24, rolling_mean_168
2026-09-13 21:38:27 | INFO     | src.feature_engineering | Feature engineering complete: 48048 rows before, 47880 rows

In [3]:
import os
import torch
torch.set_num_threads(os.cpu_count())

from src.models.lstm import LSTMForecaster
from src.training.trainer import train_model, predict

final_lstm_model = LSTMForecaster(num_features=X_train.shape[2])
final_history = train_model(final_lstm_model, X_train, y_train_scaled, X_val, y_val_scaled, epochs=20)

print("Epochs trained:", final_history["epochs_trained"])
print("Parameters:", final_history["num_parameters"])
print("Training time (s):", round(final_history["training_time_seconds"], 1))

2026-09-13 21:38:54 | INFO     | src.training.trainer | Training LSTMForecaster | 23576 parameters | device=cpu
2026-09-13 21:38:55 | INFO     | src.training.trainer |   batch 0/131
2026-09-13 21:39:25 | INFO     | src.training.trainer |   batch 100/131
2026-09-13 21:39:36 | INFO     | src.training.trainer | Epoch 1/20 | train_loss=0.056047 | val_loss=0.018835
2026-09-13 21:39:36 | INFO     | src.training.trainer |   batch 0/131
2026-09-13 21:40:03 | INFO     | src.training.trainer |   batch 100/131
2026-09-13 21:40:14 | INFO     | src.training.trainer | Epoch 2/20 | train_loss=0.008758 | val_loss=0.007238
2026-09-13 21:40:14 | INFO     | src.training.trainer |   batch 0/131
2026-09-13 21:40:59 | INFO     | src.training.trainer |   batch 100/131
2026-09-13 21:41:13 | INFO     | src.training.trainer | Epoch 3/20 | train_loss=0.004723 | val_loss=0.005487
2026-09-13 21:41:13 | INFO     | src.training.trainer |   batch 0/131
2026-09-13 21:42:12 | INFO     | src.training.trainer |   batch 1

In [4]:
from src.evaluation.metrics import evaluate_all
from src.preprocessing import inverse_transform_column
from src.config import TARGET_COL

y_pred_test_scaled = predict(final_lstm_model, X_test)
y_true_final = inverse_transform_column(y_test_scaled, scaler, scale_columns, TARGET_COL)
y_pred_final = inverse_transform_column(y_pred_test_scaled, scaler, scale_columns, TARGET_COL)

final_metrics = evaluate_all(y_true_final, y_pred_final)
print("Final LSTM:", final_metrics)

Final LSTM: {'MAE': 67.79994183849904, 'RMSE': 95.04825078148339, 'MAPE': 5.815051444068754, 'sMAPE': 5.688378138429484, 'R2': 0.7349799544628863}


In [5]:
from src.inference import save_model_artifacts

save_model_artifacts(final_lstm_model, scaler, scale_columns)

2026-09-13 21:56:13 | INFO     | src.inference | Saved model weights to C:\Users\Ram\OneDrive\Desktop\energy-demand-lstm\models\lstm_final.pt
2026-09-13 21:56:13 | INFO     | src.inference | Saved scaler to C:\Users\Ram\OneDrive\Desktop\energy-demand-lstm\models\scaler.pkl
2026-09-13 21:56:13 | INFO     | src.inference | Saved feature config to C:\Users\Ram\OneDrive\Desktop\energy-demand-lstm\models\feature_config.json


In [6]:
from src.inference import load_model_artifacts
from src.training.trainer import predict as predict_fn

loaded_model, loaded_scaler, loaded_config = load_model_artifacts()

# Confirm loaded model produces IDENTICAL predictions to the original
y_pred_original = predict_fn(final_lstm_model, X_test[:100])
y_pred_loaded = predict_fn(loaded_model, X_test[:100])

import numpy as np
print("Predictions match exactly:", np.allclose(y_pred_original, y_pred_loaded))
print("Loaded config:", loaded_config)

2026-09-13 21:56:13 | INFO     | src.inference | Loaded model artifacts successfully
Predictions match exactly: True
Loaded config: {'model_feature_columns': ['temp_avg', 'humidity_avg', 'precip_avg', 'wind_avg', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'is_weekend', 'holiday', 'school', 'lag_1', 'lag_24', 'lag_48', 'lag_168', 'rolling_mean_24', 'rolling_std_24', 'rolling_mean_168'], 'scale_columns': ['temp_avg', 'humidity_avg', 'precip_avg', 'wind_avg', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'is_weekend', 'holiday', 'school', 'lag_1', 'lag_24', 'lag_48', 'lag_168', 'rolling_mean_24', 'rolling_std_24', 'rolling_mean_168', 'nat_demand'], 'target_column': 'nat_demand', 'lookback': 72, 'horizon': 24, 'hidden_size': 64, 'num_features': 20}
